# 04 — Génération sourcée et validation des citations

**Objectif :** transformer un dossier de preuves en réponse JSON avec citations,
puis vérifier que les citations existent réellement.

Le modèle ne doit pas répondre de mémoire. Il doit utiliser uniquement les
preuves transmises.


## Étape 1 — Récupérer les preuves

On exécute la recherche multimodale pour la question AL-01.


In [1]:
from maintidoc.retrieval import Pipeline

pipeline = Pipeline.load_or_build()
retrieval = pipeline.search(
    "Que signifie le voyant orange AL-01 ?",
    "orange warning LED storage array front panel",
)
for item in retrieval["evidence"]:
    print(item["id"], "| page", item["page"], "|", item["type"])

P05-TEXT | page 5 | text
P05-TAB01 | page 5 | table
P07-TAB02 | page 7 | table
P07-TEXT | page 7 | text
P05-TEXT | page 5 | text
P07-TAB03 | page 7 | table
P05-IMG01 | page 5 | image


### Lecture du résultat

Les preuves doivent inclure le texte de la page 5, le tableau des alarmes et le
schéma FIG-04. Si ce n'est pas le cas, il faut revoir la recherche avant de
générer.


## Étape 2 — Construire le prompt

Le prompt contient :

- la question ;
- les preuves, avec identifiant et page ;
- la liste des identifiants autorisés ;
- les règles de statut et de citation.


In [2]:
from maintidoc.generation import build_prompt

prompt = build_prompt(retrieval["question"], retrieval["evidence"])
print(prompt[:2500])

Question : Que signifie le voyant orange AL-01 ?

PREUVES :
[P05-TEXT | page 5 | text]
SAN-MAN-001 | MaintiDoc | Manuel pedagogique fictif Page 5 4. Voyants, evenements et interpretation FIG-04 : signification uniquement conventionnelle dans le simulateur SAN-LAB 2000 ; ne pas extrapoler a un SAN reel. Code Etat Signification pour le simulateur Reponse documentaire AL-01 Orange Chemin FC degrade Verifier DIAG-FC-01 et FIG-02 AL-02 Orange Disque en pre-alerte Consulter FIG-03 et DISC-02 AL-03 Rouge Volume indisponible Escalade prioritaire au responsable SAN AL-04 Orange Capacite libre faible Consulter tendances et politique capacite AL-00 Vert Etat nominal declare Aucune action preventive imposee Interpretation d une photographie Exemple de question : « Le voyant orange est

[P05-TAB01 | page 5 | table]
| Code | Etat | Signification pour le simulateur | Reponse documentaire |
|---|---|---|---|
| AL-01 | Orange | Chemin FC degrade | Verifier DIAG-FC-01 et FIG-02 |
| AL-02 | Orange | Disq

### Lecture du résultat

On voit les identifiants `P05-TEXT`, `P05-TAB01`, `P05-IMG01`, etc. Le modèle doit
citer ces identifiants, pas des références documentaires comme `ALARM-TAB-01`.


## Étape 3 — Appeler Ollama

On utilise le client Ollama local. Le modèle est configurable. `format="json"`
force une sortie JSON, mais ne garantit pas la validité du contenu.

Le modèle par défaut du projet est `llama3.2:latest` ; `qwen2.5vl:7b` peut être
utilisé s'il est installé.


In [3]:
from maintidoc.generation import call_ollama

raw = call_ollama(
    "Que signifie le voyant orange AL-01 ?",
    retrieval["evidence"],
    retrieval["images"],
    model="llama3.2:latest",
)
print(raw[:1500])

{
  "status": "answered",
  "answer": "Le voyant orange AL-01 indique une dégradation du chemin FC, il est donc recommandé de vérifier le diagnostic DIAG-FC-01 et de consulter FIG-02.",
  "citations": [
    {"id": "P05-TAB01", "page": 5},
    {"id": "ALARM-TAB-01", "page": 7}
  ],
  "uncertainties": []
}


### Lecture du résultat

La sortie doit être un objet JSON avec `status`, `answer`, `citations`,
`uncertainties`. Si Ollama est indisponible, la cellule suivante utilisera le
fallback extractif.


## Étape 4 — Parser et valider

`answer_question()` fait :

1. appel du modèle ;
2. extraction du JSON ;
3. suppression des citations inconnues ;
4. validation des IDs et des pages ;
5. fallback si le modèle échoue.


In [4]:
from maintidoc.generation import answer_question

answer = answer_question(
    "Que signifie le voyant orange AL-01 ?",
    retrieval,
    model="llama3.2:latest",
)
print("Statut :", answer.get("status"))
print("Mode :", answer.get("mode"))
print("Réponse :", answer.get("answer"))
print("Citations :", answer.get("citations"))
print("Problèmes :", answer.get("issues"))

Statut : answered
Mode : ollama
Réponse : Le voyant orange AL-01 indique une dégradation du chemin FC, il est donc recommandé de vérifier le diagnostic DIAG-FC-01 et de consulter FIG-02.
Citations : [{'id': 'P05-TAB01', 'page': 5}]
Problèmes : []


### Lecture du résultat

Le statut attendu est `answered`, avec au moins une citation `P05-*` page 5.
Les citations inconnues sont retirées et signalées dans les incertitudes.


## Étape 5 — Tester le fallback

On force un modèle inexistant. Le système ne doit pas planter : il doit produire
une réponse extractive avec citation, ou une abstention.


In [5]:
fallback = answer_question(
    "Que signifie le voyant orange AL-01 ?",
    retrieval,
    model="modele-inexistant:0b",
)
print("Mode :", fallback.get("mode"))
print("Statut :", fallback.get("status"))
print("Citations :", fallback.get("citations"))
print("Raison :", fallback.get("fallback_reason"))

Mode : fallback
Statut : answered
Citations : [{'id': 'P05-TEXT', 'page': 5}]
Raison : ResponseError: model 'modele-inexistant:0b' not found (status code: 404)


### Lecture du résultat

Le mode doit être `fallback`. La réponse est volontairement moins fluide, mais
elle reste traçable. C'est une garantie de démonstration, pas une solution de
production.


## Étape 6 — Vérification sémantique : la limite

La validation actuelle est **structurelle** : l'ID et la page existent dans les
preuves.

Elle ne prouve pas que le passage soutient l'affirmation. Pour cela, il faudrait :

- une revue humaine ;
- un vérificateur séparé ;
- un juge LLM avec grille ;
- des annotations de soutien.

C'est une limite à écrire dans le rapport.


## Bilan

- La réponse est structurée en JSON.
- Les citations sont vérifiées structurellement.
- Les citations inconnues sont retirées.
- Le fallback garantit une démo même sans modèle.
- La vérification sémantique reste à faire.

**Étape suivante :** exposer l'historique et la création de ticket via MCP.
